# 07 — FastAPI Prototype
## Building API endpoints for AI Advocate System

### What is an API?
Think of API like a restaurant:
- You (client) sit at table and order food
- Waiter (API) takes your order to kitchen
- Kitchen (our AI system) prepares the food
- Waiter brings food back to you

### Endpoints we will build:
1. GET  /health        → Check if system is running
2. POST /ask           → Ask a legal question
3. POST /generate-doc  → Generate a legal document
4. POST /research      → Research a legal topic
5. GET  /stats         → System statistics

### Why FastAPI?
- Fastest Python web framework
- Auto generates documentation
- Easy to test with built-in Swagger UI
- Used by Netflix, Uber, Microsoft

In [3]:
# Cell 2
import os
import json
import time
import faiss
import numpy as np
import nest_asyncio
from groq import Groq
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

# Apply nest_asyncio for running in notebook
nest_asyncio.apply()

# Load environment
load_dotenv('../.env', override=True)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# Connect Groq
groq_client = Groq(api_key=GROQ_API_KEY)

# Load FAISS
index = faiss.read_index('../vector_store/legal_index.faiss')

# Load metadata
with open('../vector_store/metadata.json', 
          'r', encoding='utf-8') as f:
    chunks = json.load(f)

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Groq connected!")
print(f"✅ FAISS loaded: {index.ntotal} vectors")
print(f"✅ Chunks loaded: {len(chunks)}")
print("✅ Embedding model loaded!")
print()
print("🚀 Ready to build FastAPI!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5710.31it/s]


✅ Groq connected!
✅ FAISS loaded: 56617 vectors
✅ Chunks loaded: 56617
✅ Embedding model loaded!

🚀 Ready to build FastAPI!


# Why are we installing in notebook instead of terminal?
Because your Python in VS Code notebook uses a different environment than the terminal sometimes. Installing inside notebook guarantees it installs in the correct environment! ✅

In [2]:
# Cell 3
# Install FastAPI and related libraries
!pip install fastapi uvicorn pydantic nest-asyncio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\mrige\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## cHECKING Notebook Python Path and Terminal PATH 
BOTH ARE SAME

In [4]:
# Cell 3
import sys
print("Notebook Python:", sys.executable)

Notebook Python: C:\Users\mrige\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


## Step 1 — Define Request and Response Models
Pydantic models define exact shape of API requests

In [5]:
# Cell 5
# Define request and response models
from pydantic import BaseModel
from typing import Optional, List

# Request Models (what user sends TO api)
class QuestionRequest(BaseModel):
    question: str
    k: Optional[int] = 5  # number of sections to retrieve

class DocumentRequest(BaseModel):
    doc_type: str
    party1: str
    party2: str
    terms: str
    extra_details: Optional[str] = ""

class ResearchRequest(BaseModel):
    topic: str

# Response Models (what api sends BACK to user)
class QuestionResponse(BaseModel):
    question: str
    answer: str
    sources: List[str]
    status: str

class DocumentResponse(BaseModel):
    doc_type: str
    document: str
    saved_to: str
    status: str

class ResearchResponse(BaseModel):
    topic: str
    research: str
    status: str

print("✅ Request models defined:")
print("   → QuestionRequest")
print("   → DocumentRequest")
print("   → ResearchRequest")
print()
print("✅ Response models defined:")
print("   → QuestionResponse")
print("   → DocumentResponse")
print("   → ResearchResponse")

✅ Request models defined:
   → QuestionRequest
   → DocumentRequest
   → ResearchRequest

✅ Response models defined:
   → QuestionResponse
   → DocumentResponse
   → ResearchResponse


## Step 2 — Build FastAPI App
Define all API endpoints

In [6]:
# Cell 7
from fastapi import FastAPI
from fastapi.responses import JSONResponse

# Create FastAPI app
app = FastAPI(
    title="AI Advocate API",
    description="RAG based Legal Assistant API for Indian Law",
    version="1.0.0"
)

# Helper functions
def retrieve_sections(question, k=5):
    query_vector = embedding_model.encode(
        [question]).astype('float32')
    distances, indices = index.search(query_vector, k=k)
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        results.append({
            'act'    : chunk['act_title'],
            'section': chunk['section_id'],
            'heading': chunk['section_heading'],
            'text'   : chunk['text']
        })
    return results

def call_groq(prompt):
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system",
                     "content": "You are an expert "
                                "Indian legal assistant."},
                    {"role": "user",
                     "content": prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            wait_time = (attempt + 1) * 10
            time.sleep(wait_time)
    return "Service temporarily unavailable."

# ===== ENDPOINT 1 — Health Check =====
@app.get("/health")
def health_check():
    return {
        "status"  : "running",
        "message" : "AI Advocate API is live!",
        "version" : "1.0.0",
        "vectors" : index.ntotal,
        "chunks"  : len(chunks)
    }

# ===== ENDPOINT 2 — Ask Legal Question =====
@app.post("/ask", response_model=QuestionResponse)
def ask_question(req: QuestionRequest):
    try:
        # Retrieve sections
        results = retrieve_sections(req.question, req.k)
        
        # Build context
        context = ""
        sources = []
        for r in results:
            context += f"\nAct: {r['act']}\n"
            context += f"Section: {r['section']} - "
            context += f"{r['heading']}\n"
            context += f"Text: {r['text']}\n"
            sources.append(
                f"{r['act']} — {r['section']} {r['heading']}")
        
        # Generate answer
        prompt = f"""Answer this legal question based on 
Indian law sections below.
Cite exact Act and Section.

LAW SECTIONS:
{context}

QUESTION: {req.question}"""
        
        answer = call_groq(prompt)
        
        return QuestionResponse(
            question = req.question,
            answer   = answer,
            sources  = sources,
            status   = "success"
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

# ===== ENDPOINT 3 — Generate Document =====
@app.post("/generate-doc", response_model=DocumentResponse)
def generate_document(req: DocumentRequest):
    try:
        prompt = f"""Draft a professional {req.doc_type} 
under Indian law.
Party 1: {req.party1}
Party 2: {req.party2}
Terms: {req.terms}
Extra: {req.extra_details}

Include all standard legal clauses."""
        
        document = call_groq(prompt)
        
        # Save document
        filename = f"../outputs/{req.doc_type.replace(' ','_')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(document)
        
        return DocumentResponse(
            doc_type = req.doc_type,
            document = document,
            saved_to = filename,
            status   = "success"
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

# ===== ENDPOINT 4 — Research Topic =====
@app.post("/research", response_model=ResearchResponse)
def research_topic(req: ResearchRequest):
    try:
        prompt = f"""Research this Indian legal topic:
{req.topic}

Include relevant Acts, Sections and practical implications."""
        
        research = call_groq(prompt)
        
        return ResearchResponse(
            topic    = req.topic,
            research = research,
            status   = "success"
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

# ===== ENDPOINT 5 — System Stats =====
@app.get("/stats")
def get_stats():
    return {
        "total_vectors"  : index.ntotal,
        "total_chunks"   : len(chunks),
        "embedding_model": "all-MiniLM-L6-v2",
        "llm_model"      : "llama-3.3-70b-versatile",
        "llm_provider"   : "Groq",
        "documents_types": [
            "Rental Agreement",
            "NDA",
            "Employment Contract",
            "Partnership Deed",
            "Service Agreement"
        ],
        "status": "operational"
    }

print("✅ FastAPI app created!")
print()
print("📋 Endpoints defined:")
print("   GET  /health      → Check system status")
print("   POST /ask         → Ask legal question")
print("   POST /generate-doc → Generate legal document")
print("   POST /research    → Research legal topic")
print("   GET  /stats       → System statistics")

✅ FastAPI app created!

📋 Endpoints defined:
   GET  /health      → Check system status
   POST /ask         → Ask legal question
   POST /generate-doc → Generate legal document
   POST /research    → Research legal topic
   GET  /stats       → System statistics


## Step 3 — Run FastAPI Server
Starting the server inside notebook using nest_asyncio
Then testing all endpoints

In [7]:
# Cell 9
import threading
import uvicorn

# Run server in background thread
def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="warning"
    )

# Start server in background
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait for server to start
time.sleep(3)

print("✅ FastAPI server is running!")
print()
print("🌐 Server URL    : http://127.0.0.1:8000")
print("📚 API Docs      : http://127.0.0.1:8000/docs")
print("📋 Alternative   : http://127.0.0.1:8000/redoc")
print()
print("Open any URL above in your browser!")

✅ FastAPI server is running!

🌐 Server URL    : http://127.0.0.1:8000
📚 API Docs      : http://127.0.0.1:8000/docs
📋 Alternative   : http://127.0.0.1:8000/redoc

Open any URL above in your browser!


## Final Summary — FastAPI Complete

In [ ]:
# Cell 11
print("=" * 60)
print("📊 FASTAPI PROTOTYPE SUMMARY")
print("=" * 60)
print()
print("✅ Framework  : FastAPI 0.136.1")
print("✅ Server     : Uvicorn 0.46.0")
print("✅ URL        : http://127.0.0.1:8000")
print("✅ Docs URL   : http://127.0.0.1:8000/docs")
print()
print("Endpoints Built:")
print("   ✅ GET  /health      → System status")
print("   ✅ POST /ask         → Legal questions")
print("   ✅ POST /generate-doc → Legal documents")
print("   ✅ POST /research    → Legal research")
print("   ✅ GET  /stats       → System statistics")
print()
print("Tested Successfully:")
print("   ✅ /health → status running, 56617 vectors")
print("   ✅ /ask    → IPC Section 302 murder answer")
print()
print("Key Features:")
print("   ✅ Pydantic validation")
print("   ✅ Auto Swagger UI docs")
print("   ✅ JSON responses")
print("   ✅ Error handling with HTTPException")
print("   ✅ Retry logic for Groq")
print()
print("=" * 60)
print("✅ Ready to move to 08_evaluation.ipynb!")
print("=" * 60)

📊 FASTAPI PROTOTYPE SUMMARY

✅ Framework  : FastAPI 0.136.1
✅ Server     : Uvicorn 0.46.0
✅ URL        : http://127.0.0.1:8000
✅ Docs URL   : http://127.0.0.1:8000/docs

Endpoints Built:
   ✅ GET  /health      → System status
   ✅ POST /ask         → Legal questions
   ✅ POST /generate-doc → Legal documents
   ✅ POST /research    → Legal research
   ✅ GET  /stats       → System statistics

Tested Successfully:
   ✅ /health → status running, 56617 vectors
   ✅ /ask    → IPC Section 302 murder answer

Key Features:
   ✅ Pydantic validation
   ✅ Auto Swagger UI docs
   ✅ JSON responses
   ✅ Error handling with HTTPException
   ✅ Retry logic for Groq

✅ Ready to move to 08_evaluation.ipynb!


## Challenges & Solutions in 07_fastapi_prototype.ipynb

---

### Challenge 1 — FastAPI Not Installed
**Error:**
ModuleNotFoundError: No module named 'fastapi'

**Reason:**
- FastAPI not installed in Python environment
- pip install was not run before importing

**Solution:**
- Installed inside notebook using !pip install
- !pip install fastapi uvicorn pydantic nest-asyncio
- Successfully installed fastapi-0.136.1

**Lesson Learned:**
- Always install libraries before importing
- Use !pip install inside notebook for guaranteed
  correct Python environment installation

---

### Challenge 2 — Running Server Inside Notebook
**Problem:**
- FastAPI normally runs as standalone server
- Cannot run blocking uvicorn.run() in notebook
- Would freeze the entire notebook

**Solution:**
- Used threading.Thread to run server in background
- Used nest_asyncio.apply() to allow async in notebook
- Server runs in daemon thread — stops when notebook stops

**Lesson Learned:**
- nest_asyncio is essential for async code in notebooks
- daemon=True means thread dies when main program exits
- Threading allows parallel execution in notebooks

---

### Challenge 3 — Wrong Default Request Body
**Problem:**
- Swagger UI shows default "string" in request body
- Sending default body gives irrelevant answer
- Users confused about correct request format

**Solution:**
- Always replace default "string" with proper JSON
- Example for /ask endpoint:
  {"question": "What is punishment for murder?", "k": 5}
- Pydantic models ensure correct format

**Lesson Learned:**
- Swagger UI default values are just placeholders
- Always test with real meaningful data
- Pydantic BaseModel enforces correct request format

---

### Challenge 4 — Poor Answer Quality
**Problem:**
- /ask endpoint returned irrelevant law sections
- FAISS retrieving military acts for general questions
- Answer quality depends on retrieval quality

**Solution:**
- This is expected without HyDE implementation
- Basic retrieval works but needs improvement
- HyDE upgrade will significantly improve quality

**Lesson Learned:**
- API quality = RAG quality
- Better retrieval = better API answers
- Future upgrade: add HyDE to /ask endpoint

---

### Overall Interview Answer:
> "We built a FastAPI microservice with 5 endpoints
> — /health, /ask, /generate-doc, /research and /stats.
> The server runs on uvicorn with threading to work
> inside Jupyter notebooks. FastAPI automatically
> generates Swagger UI at /docs — allowing developers
> to test all endpoints interactively without writing
> any code. We tested /health returning 56,617 vectors
> confirmed and /ask correctly citing IPC Section 302
> for murder punishment query with 200 status code."